In [2]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/clean_modules_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: future


Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths


The following objects are masked from ‘package:data.table’:

    dcast, melt




In [3]:
mod_def <- "PosBC"

### Prep enrichments

In [1]:
network_dir <- "GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules"

In [5]:
data_source <- "GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules_MO_2117sets_enrichments"

top_mods_df <- fread(paste0("data/enrichments/", data_source, ".csv"), data.table=FALSE)
top_mods_df$Cell_type <- gsub("/", "_", top_mods_df$Cell_type, fixed=TRUE)
top_mods_df$Cell_type <- gsub(" ", "_", top_mods_df$Cell_type, fixed=TRUE)

### Prep bulk data

In [6]:
# Bulk gene expression should be the same dataset used in to find modules and enrichments

bulk_expr <- fread(paste0("data/GTEx_cortex_counts_All_501_outliers_removed_ComBat_SMGEBTCH_corrected.csv"), data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"

### Get enriched module genes

In [8]:
top_mods_df %>%
    arrange(Qval) %>%
    select(Cell_type, Qval) %>%
    head(n=20)

,Cell_type,Qval
,<chr>,<dbl>
1,NAGY_2020_EX1,0.000000e+00
2,MORABITO_2021_EX5_DE_SUBTYPE_CLUSTERS,0.000000e+00
3,AGARWAL_CTX_2020_ASTROCYTE,0.000000e+00
4,CAJIGAS_UP_IN_CA1_NEUROPIL,0.000000e+00
5,SAM_GLIOMA_TURQUOISE_GENES_IDH1_MODULE_6302GENES,0.000000e+00
6,AGARWAL_SN_2020_DANS,0.000000e+00
7,LAU_2020_MICROGLIA,6.926646e-286
8,Layer5_He_NatNeuro_2017,1.643199e-258
9,kelley_microglia,2.259857e-236


In [7]:
sets <- c("Micro_PVM", "Oligo", "Astro", "VLMC")

top_mods_df_subset <- top_mods_df[top_mods_df$Cell_type %in% sets,]

## Remove module genes from data

In [ ]:
expr_filtered <- remove_mod_genes(expr, top_mods_df_subset, mod_def) 

dim(expr_fitlered)

[1] 51923   502

## Regress out module signals

In [ ]:
# Make data frame of module eigengenes

mod_eigs_df <- get_mod_eigs(top_mods_df_subset)

# Make sure expression data and ME data samples match

mod_eigs_df <- mod_eigs_df[match(colnames(expr_filtered)[-1], mod_eigs_df$Sample),]
all.equal(mod_eigs_df$Sample, colnames(expr_filtered)[-1])

In [ ]:
# Inspect corr between MEs
mod_eigs_mat <- mod_eigs_df[,-1]
cor(mod_eigs_mat)

In [ ]:
expr_adj <- regress_out_signals(expr_fitlered, mod_eigs_df, gene_set_names=sets)

In [ ]:
fwrite(expr_adj, file=paste0("data/cleaned/", data_source, "_", paste(sets, collapse="_"), "_", mod_def, "_cleaned.csv"))